# Lab 24-04 · ADC Bits, MLSD and FEC

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
import dsp

## Part 1 · ADC bits

PAM4 through $h$ = (0.1, 1, 0.5, 0.2), noise 0.075 rms, an 11-tap MMSE FFE and a ±1.8 full scale. *Predict* the symbol error rate from 4 to 8 bits.

In [2]:
h = {-1: 0.1, 0: 1.0, 1: 0.5, 2: 0.2}
sigma = 0.075
a = dsp.pam_symbols(200_000, 4, seed=5)
y = dsp.channel(a, h, sigma, seed=6)
c = dsp.mmse_ffe(h, sigma, 2, 8, 5 / 9)
for bits in (4, 5, 6, 7, 8, None):
    yq = y if bits is None else dsp.quantize(y, bits, 3.6)
    ser = np.mean(dsp.slice_pam(dsp.apply_ffe(yq, c))[20:-20] != a[20:-20])
    label = "ideal" if bits is None else f"{bits} bits (SQNR {dsp.sqnr_db(bits, 3.6):.1f} dB)"
    print(f"{label:24s} SER {ser:.2e}")

4 bits (SQNR 21.2 dB)    SER 3.94e-03
5 bits (SQNR 27.2 dB)    SER 7.15e-04
6 bits (SQNR 33.2 dB)    SER 3.05e-04
7 bits (SQNR 39.3 dB)    SER 2.50e-04
8 bits (SQNR 45.3 dB)    SER 2.60e-04
ideal                    SER 2.45e-04


## Part 2 · MLSD against DFE

*Predict* the closest error events for $h = (1, \alpha)$ and the SNR gap at a BER of 10⁻³ on $h = (1, 1)$.

In [3]:
for alpha in (0.5, 1.0, 1.25):
    d, e = dsp.min_error_distance([1.0, alpha])
    print(f"alpha {alpha}: d_min^2 = {d:.2f} from error event {e}; gain over DFE {10*np.log10(d/4):.2f} dB")
sigmas = np.array([0.32, 0.36, 0.40, 0.44, 0.48])
a2 = dsp.pam_symbols(200_000, 2, seed=7)
ber_dfe, ber_mlsd = [], []
for i, s in enumerate(sigmas):
    y2 = dsp.channel(a2, {0: 1.0, 1: 1.0}, s, seed=10 + i)
    ber_dfe.append(np.mean(dsp.dfe_ideal(y2, a2, [1.0, 1.0]) != a2))
    ber_mlsd.append(np.mean(dsp.viterbi(y2, [1.0, 1.0]) != a2))
    print(f"sigma {s:.2f}: DFE {ber_dfe[-1]:.2e} (theory {norm.sf(1/s):.2e}), MLSD {ber_mlsd[-1]:.2e}")
s_dfe = 1 / norm.isf(1e-3)
s_mlsd = np.exp(np.interp(np.log(1e-3), np.log(ber_mlsd), np.log(sigmas)))
print(f"BER 1e-3: DFE at sigma {s_dfe:.3f}, MLSD at {s_mlsd:.3f}: gap {20*np.log10(s_mlsd/s_dfe):.2f} dB")
plt.figure()
plt.semilogy(20 * np.log10(1 / sigmas), ber_dfe, "o-", label="ideal DFE (simulated)")
plt.semilogy(20 * np.log10(1 / sigmas), ber_mlsd, "s-", label="MLSD (simulated)")
plt.xlabel("SNR = 20 log10(h0/sigma) (dB)"); plt.ylabel("BER"); plt.legend(); plt.grid(True, which="both")
plt.show()

alpha 0.5: d_min^2 = 5.00 from error event (-2.0,); gain over DFE 0.97 dB
alpha 1.0: d_min^2 = 8.00 from error event (-2.0,); gain over DFE 3.01 dB
alpha 1.25: d_min^2 = 10.25 from error event (-2.0,); gain over DFE 4.09 dB


sigma 0.32: DFE 9.25e-04 (theory 8.89e-04), MLSD 5.00e-06


sigma 0.36: DFE 2.72e-03 (theory 2.74e-03), MLSD 1.40e-04


sigma 0.40: DFE 6.14e-03 (theory 6.21e-03), MLSD 9.50e-04


sigma 0.44: DFE 1.12e-02 (theory 1.15e-02), MLSD 2.11e-03


sigma 0.48: DFE 1.87e-02 (theory 1.86e-02), MLSD 5.09e-03
BER 1e-3: DFE at sigma 0.324, MLSD at 0.402: gap 1.89 dB


## Part 3 · FEC curves

*Predict* RS(544,514)'s output BER at an input of 2.4 × 10⁻⁴.

In [4]:
pin = np.logspace(-5, -2.7, 40)
pout = [dsp.rs_ber_out(p) for p in pin]
print(f"at 2.4e-4: UCR {dsp.rs_ucr(2.4e-4):.3e}, output BER {dsp.rs_ber_out(2.4e-4):.3e}")
print(f"at 4.8e-4: output BER {dsp.rs_ber_out(4.8e-4):.3e}")
plt.figure()
plt.loglog(pin, np.maximum(pout, 1e-30))
plt.axvline(2.4e-4, color="0.5", ls=":")
plt.xlabel("input BER (random errors)"); plt.ylabel("output BER"); plt.grid(True, which="both")
plt.show()

at 2.4e-4: UCR 8.152e-13, output BER 2.398e-15
at 4.8e-4: output BER 4.723e-11


## Part 4 · Rates and latency

In [5]:
r = 106.25 * 128 / 120 * (1 + 384 / (3264 * 128))
print(f"inner-code line rate {r:.4f} GBd")
for pin_, rate, name in ((2.2e-4, 514 / 544, "KP4"), (4.85e-3, 514 / 544 * 120 / 128, "concatenated")):
    print(f"{name:12s} NCG {dsp.ncg_db(pin_, 1e-13, rate):.2f} dB at 1e-13, {dsp.ncg_db(pin_, 1e-15, rate):.2f} dB at 1e-15")
print(f"codeword time: {dsp.codeword_time(5440, 106.25e9)*1e9:.1f} ns at 106.25 Gb/s, "
      f"{dsp.codeword_time(5440, 212.5e9)*1e9:.1f} ns at 212.5 Gb/s; flit x16 {2048/(16*64e9)*1e9:.1f} ns")
print(f"longest burst corrected: {dsp.burst_corrected()} bits; with 4-way interleave {dsp.burst_corrected(interleave=4)} bits")

inner-code line rate 113.4375 GBd
KP4          NCG 6.16 dB at 1e-13, 6.83 dB at 1e-15
concatenated NCG 8.54 dB at 1e-13, 9.22 dB at 1e-15
codeword time: 51.2 ns at 106.25 Gb/s, 25.6 ns at 212.5 Gb/s; flit x16 2.0 ns
longest burst corrected: 141 bits; with 4-way interleave 591 bits
